In [1]:
from itertools import product
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scaling.visualize import visualize_train_curves, plot_line_fit, plot_isoflops
from pathlib import Path
from scaling.utils import (
    get_pareto_frontier,
    get_final_points_from_curve_set,
    fit_linear_model,
    functional_form_chin3,
    fit_parametric_form,
    fit_parametric_form_stable,
    functional_form_chin3_stable,
)

In [44]:
unique_col_list = ["base_N", "target_N", "tkpm", "shrink", "method"]
y_col = "Validation Loss"
x_col = "flops"


def preprocess_warmstarting(df, y_col_to_smooth=None, smoothing_window=100):
    __df = pd.DataFrame()
    for i, x in enumerate(df.groupby(unique_col_list)):
        _df = x[1].sort_values(by="flops")
        # smooth it
        if y_col_to_smooth is not None:
            # +"_smoothed"
            _df[y_col_to_smooth] = _df[y_col_to_smooth].rolling(smoothing_window, win_type='gaussian', min_periods=1).mean(std=smoothing_window / 10)
        
        # scaling tokens and flops to the max
        max_intended_tokens = (_df.iloc[-1]["target_N"] * _df.iloc[-1]["tkpm"])
        if abs((max_intended_tokens -  _df["tokens"].max()) / _df["tokens"].max()) > 0.01:
            print("Wrong tkpm: ", x[0])
            continue
        _df["tokens"] = np.round(max_intended_tokens / _df["tokens"].max() * _df["tokens"])
        
        max_intended_flops = 6. * max_intended_tokens * _df["target_N"]
        _df["flops"] = np.round(max_intended_flops / _df["flops"].max() * _df["flops"])
        
        __df = pd.concat([__df, _df])
    
    print('Droping tkpm <= 5')    
    __df = __df[__df['tkpm'] > 5.]
    
    return __df

In [45]:
warmstarting_df = pd.read_parquet(
    "../data/warmstart_runs_flattened.parquet",
)
warmstarting_df = warmstarting_df.dropna(subset=[y_col])
warmstarting_df = preprocess_warmstarting(warmstarting_df)

Wrong tkpm:  (77124608, 1217722368, 10.0, 0.0, 'mup')
Droping tkpm <= 5


In [ ]:
mup_df = warmstarting_df[warmstarting_df['method']=='mup']
df = get_final_points_from_curve_set(
    mup_df,
    unique_col_list,
    x_col="flops",
    y_col="Validation Loss",
    get_pareto=False,
)

N, _ = df["target_N"].values, df["tokens"].values
_D = df["target_N"] * df["tkpm"]
y = df["Validation Loss"].values

_df = pd.DataFrame.from_dict({
    "N": N,
    "D": _D,
    "Loss": y
}).groupby(by=["N", "D"]).min().reset_index()
_df.sort_values(by=["N", "D"], inplace=True)

data_X = _df[["N", "D"]].values
data_y = _df["Loss"].values